# Clase 9 — Unidad 9: Procesamiento de Imágenes en Investigación Biomédica

**Curso:** Introducción al Análisis de Datos — 2026, Segundo Semestre
**Material de referencia:** *Python Essentials for Biomedical Data Analysis* — Capítulo 9, "Image Processing in Biomedical Research"

## Objetivos de aprendizaje
- Comprender cómo se **representa** una imagen digital (píxeles, escala de grises vs RGB, profundidad de bits).
- Aplicar **preprocesamiento**: reducción de ruido, normalización y realce de contraste.
- Realizar **segmentación** por umbralización (Otsu) y **detección de bordes** (Sobel, Canny).
- **Extraer características** de objetos: etiquetar regiones y medir área, perímetro, etc.
- Conocer las principales **librerías** de imágenes en Python y los desafíos éticos del área.

## Contenidos de la clase
1. El rol del procesamiento de imágenes en biomedicina
2. Representación de imágenes
3. Preprocesamiento de imágenes
4. Segmentación
5. Extracción de características
6. Librerías de Python para imágenes
7. Desafíos y ética
8. Ejercicios de práctica

> Esta clase corresponde a la **Unidad 9** del libro guía. Usaremos una **imagen real de inmunohistoquímica (IHC)** incluida en scikit-image y una **imagen sintética de células** para practicar segmentación y conteo. Todo se ejecuta sin archivos externos.

## Preparación del entorno

Usaremos **scikit-image** (`skimage`), la librería de procesamiento de imágenes del ecosistema científico de Python, junto con numpy, scipy y matplotlib.

In [ ]:
%pip install scikit-image numpy scipy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.color import rgb2gray
from skimage.draw import disk

# 1) Imagen real de inmunohistoquímica (IHC), a color (incluida en scikit-image)
ihc = data.immunohistochemistry()          # arreglo RGB
ihc_gris = rgb2gray(ihc)                     # versión en escala de grises (float 0..1)

# 2) Imagen sintética de "células" (para segmentar y contar)
np.random.seed(9)
celulas = np.zeros((220, 220))
centros = [(45, 55, 16), (60, 150, 14), (120, 80, 20), (160, 160, 12),
           (90, 30, 10), (170, 60, 15), (40, 180, 13)]
for (fila, col, radio) in centros:
    rr, cc = disk((fila, col), radio, shape=celulas.shape)
    celulas[rr, cc] = 0.9
celulas = np.clip(celulas + np.random.normal(0, 0.08, celulas.shape), 0, 1)

AZUL, NARANJO = "#0072B2", "#E69F00"
print("IHC (color):", ihc.shape, "| IHC (gris):", ihc_gris.shape)
print("Imagen de células:", celulas.shape, "| N° de células dibujadas:", len(centros))

## 1. El rol del procesamiento de imágenes en biomedicina

El **procesamiento de imágenes** extrae información útil de imágenes médicas (rayos X, resonancia, tomografía, microscopía, histopatología). Ayuda en el **diagnóstico**, la **planificación de tratamientos** y la **investigación**: detectar tumores, contar y clasificar células, delinear órganos, cuantificar cambios en tejidos, etc.

Los desafíos incluyen la **variabilidad** entre modalidades y equipos, la **complejidad** de las estructuras biológicas, la **alta dimensionalidad** (imágenes 3D/4D, whole-slide) y la necesidad de **exactitud** por su impacto clínico.

## 2. Representación de imágenes

Una imagen digital es un **arreglo de números**. Cada valor es un **píxel**:
- **Escala de grises:** cada píxel es una intensidad (0 = negro … 255 = blanco en 8 bits).
- **Color (RGB):** cada píxel tiene 3 componentes (rojo, verde, azul).
- **Profundidad de bits:** 8 bits → 256 niveles; 16 bits → 65 536 niveles.
- **Imagen binaria:** solo 2 valores (0 = fondo, 1 = objeto).

In [ ]:
# Mostrar la imagen IHC a color y en escala de grises
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(ihc)
ax[0].set_title("IHC — color (RGB)")
ax[0].axis("off")
ax[1].imshow(ihc_gris, cmap="gray")
ax[1].set_title("IHC — escala de grises")
ax[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Una imagen ES un arreglo de NumPy
print("Forma de la imagen a color:", ihc.shape, "(alto, ancho, canales RGB)")
print("Forma en escala de grises:", ihc_gris.shape, "(alto, ancho)")
print("Tipo y rango (color):", ihc.dtype, "->", ihc.min(), "a", ihc.max())
print("Rango en grises (float):", round(ihc_gris.min(), 2), "a", round(ihc_gris.max(), 2))

print("\nUn parche de 5x5 píxeles (valores de intensidad):")
print(ihc_gris[100:105, 100:105].round(2))

## 3. Preprocesamiento de imágenes

El preprocesamiento mejora la calidad de la imagen antes del análisis.

### 3.1 Reducción de ruido

El **ruido** son variaciones no deseadas en la intensidad. Filtros comunes:
- **Gaussiano:** suaviza promediando con los vecinos (reduce ruido, difumina un poco).
- **Mediana:** reemplaza cada píxel por la mediana de su vecindario (preserva mejor los bordes).

In [ ]:
from skimage.filters import gaussian, median
from skimage.morphology import disk as disco

# Agregamos ruido a la imagen para luego filtrarlo
np.random.seed(0)
ruidosa = np.clip(ihc_gris + np.random.normal(0, 0.12, ihc_gris.shape), 0, 1)

filtrada_gauss = gaussian(ruidosa, sigma=1)
filtrada_mediana = median(ruidosa, disco(3))

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, img, t in zip(ax, [ruidosa, filtrada_gauss, filtrada_mediana],
                     ["Con ruido", "Filtro Gaussiano", "Filtro de mediana"]):
    a.imshow(img, cmap="gray"); a.set_title(t); a.axis("off")
plt.tight_layout()
plt.show()

### 3.2 Normalización

Ajusta las intensidades a un rango estándar, útil cuando las imágenes vienen de fuentes distintas.
- **Min-Max:** escala los valores a [0, 1].
- **Z-score:** media 0 y desviación 1.

In [ ]:
min_max = (ihc_gris - ihc_gris.min()) / (ihc_gris.max() - ihc_gris.min())
z_score = (ihc_gris - ihc_gris.mean()) / ihc_gris.std()

print("Min-Max -> rango:", round(min_max.min(), 2), "a", round(min_max.max(), 2))
print("Z-score -> media:", round(z_score.mean(), 2), "| desviación:", round(z_score.std(), 2))

### 3.3 Realce de contraste

Redistribuye las intensidades para hacer más visibles las estructuras.
- **Ecualización de histograma:** mejora el contraste global.
- **CLAHE** (ecualización adaptativa con límite de contraste): mejora el contraste **por regiones**, sin amplificar demasiado el ruido.

In [ ]:
from skimage import exposure

eq_global = exposure.equalize_hist(ihc_gris)
eq_clahe = exposure.equalize_adapthist(ihc_gris, clip_limit=0.03)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, img, t in zip(ax, [ihc_gris, eq_global, eq_clahe],
                     ["Original", "Ecualización global", "CLAHE (adaptativa)"]):
    a.imshow(img, cmap="gray"); a.set_title(t); a.axis("off")
plt.tight_layout()
plt.show()

## 4. Segmentación

Segmentar es **particionar la imagen en regiones u objetos** (células, órganos, tumores).

### 4.1 Umbralización (thresholding)

Convierte una imagen en grises a **binaria**: los píxeles se marcan como objeto o fondo según su intensidad. El método de **Otsu** elige automáticamente el umbral óptimo. Lo aplicamos a la imagen de células.

In [ ]:
from skimage.filters import threshold_otsu

umbral = threshold_otsu(celulas)
binaria = celulas > umbral   # imagen binaria: True = célula, False = fondo

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(celulas, cmap="gray"); ax[0].set_title("Original (células)"); ax[0].axis("off")

ax[1].hist(celulas.ravel(), bins=50, color=AZUL)
ax[1].axvline(umbral, color=NARANJO, linewidth=2, label=f"Umbral Otsu = {umbral:.2f}")
ax[1].set_title("Histograma de intensidades"); ax[1].set_xlabel("Intensidad"); ax[1].legend()

ax[2].imshow(binaria, cmap="gray"); ax[2].set_title("Imagen binaria (Otsu)"); ax[2].axis("off")
plt.tight_layout()
plt.show()

### 4.2 Detección de bordes

Los **bordes** marcan las fronteras de las estructuras (cambios bruscos de intensidad).
- **Sobel:** calcula el gradiente de intensidad.
- **Canny:** método más sofisticado, con reducción de ruido incorporada.

In [ ]:
from skimage.filters import sobel
from skimage.feature import canny

bordes_sobel = sobel(ihc_gris)
bordes_canny = canny(ihc_gris, sigma=2)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, img, t in zip(ax, [ihc_gris, bordes_sobel, bordes_canny],
                     ["Original", "Bordes — Sobel", "Bordes — Canny"]):
    a.imshow(img, cmap="gray"); a.set_title(t); a.axis("off")
plt.tight_layout()
plt.show()

## 5. Extracción de características

Una vez segmentada la imagen, podemos **etiquetar** cada objeto y **medir** sus propiedades (área, perímetro, forma). Esto permite, por ejemplo, **contar células** y cuantificar su tamaño.

In [ ]:
from skimage.measure import label, regionprops
from skimage.color import label2rgb

# Etiquetar cada región conectada de la imagen binaria
etiquetas = label(binaria)
n_objetos = etiquetas.max()
print("Número de objetos (células) detectados:", n_objetos)

# Medir propiedades de cada objeto
print("\nPropiedades de cada célula:")
for region in regionprops(etiquetas):
    print(f"  Célula {region.label}: área={region.area} px | "
          f"perímetro={region.perimeter:.1f} | excentricidad={region.eccentricity:.2f}")

# Visualizar cada célula con un color distinto
plt.figure(figsize=(5, 5))
plt.imshow(label2rgb(etiquetas, bg_label=0))
plt.title(f"Células etiquetadas (n = {n_objetos})")
plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Librerías de Python para imágenes

| Librería | Para qué sirve |
|---|---|
| **scikit-image** (`skimage`) | Análisis científico de imágenes (filtros, segmentación, medición). La usamos en esta clase. |
| **OpenCV** (`cv2`) | Visión por computador de propósito general y en tiempo real. |
| **SimpleITK** | Imágenes médicas 2D/3D/4D; **registro** y segmentación; lee DICOM, NIfTI. |
| **Mahotas** | Operaciones morfológicas, watershed y análisis de textura; muy rápida. |
| **Napari** | Visor **interactivo** multidimensional (2D/3D/4D) para grandes datasets. |
| **pydicom** | Leer archivos **DICOM** (el estándar de imágenes médicas) y sus metadatos (visto en la Unidad 3). |

## 7. Desafíos y ética

- **Variabilidad de los datos:** distintas modalidades y equipos producen imágenes con calidad, resolución y contraste diferentes.
- **Complejidad biológica:** órganos, células y tumores varían mucho en forma y tamaño; pueden solaparse.
- **Alta dimensionalidad:** una sola imagen histopatológica puede tener miles de millones de píxeles.
- **Exactitud y confiabilidad:** los errores impactan diagnósticos y tratamientos.
- **Ética y privacidad:** las imágenes contienen información sensible; los modelos de IA pueden tener **sesgos** si se entrenan con datos no representativos, y deben ser **explicables**.

## 8. Ejercicios de práctica

Usa las imágenes `ihc_gris` y `celulas` (o carga otra con `skimage.data`) para resolver:

1. **Representación:** carga la imagen `data.coins()` de scikit-image, muestra su forma y su tipo de dato, y visualízala.
2. **Histograma:** grafica el histograma de intensidades de `ihc_gris`. ¿En qué rango se concentran los valores?
3. **Reducción de ruido:** agrega ruido a `celulas` y compáralo con el resultado de un filtro gaussiano.
4. **Realce de contraste:** aplica ecualización de histograma a `ihc_gris` y muéstrala junto a la original.
5. **Umbralización:** aplica Otsu a `data.coins()` y muestra la imagen binaria resultante.
6. **Bordes:** aplica detección de bordes de Canny a `celulas`.
7. **Conteo (desafío):** segmenta `data.coins()` con Otsu, etiqueta las regiones y **cuenta cuántas monedas** hay. (Pista: puede que necesites limpiar objetos pequeños.)

### Preguntas para reflexionar
1. ¿Por qué una imagen en escala de grises se representa como un arreglo 2D y una a color como 3D?
2. ¿Qué ventaja tiene el filtro de mediana sobre el gaussiano para preservar bordes?
3. ¿Por qué el método de Otsu es útil cuando no conocemos el umbral adecuado?
4. ¿Para qué sirve etiquetar regiones tras la segmentación?
5. ¿Qué riesgos éticos existen al usar IA sobre imágenes médicas de pacientes?

In [ ]:
# Espacio para resolver los ejercicios
